In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv(r"C:\Users\Sairaj\OneDrive\Desktop\Movie Recommendation Project\Datasets\movies.csv")

In [3]:
df.shape

(62423, 3)

In [4]:
dt = pd.read_csv(r"C:\Users\Sairaj\OneDrive\Desktop\Movie Recommendation Project\Datasets\ratings.csv")

In [5]:
dt.shape

(25000095, 4)

In [6]:
dataset = pd.merge(df, dt, on = "movieId", how = "left")

In [7]:
dataset.shape

(25003471, 6)

In [8]:
dataset.head()

,movieId,title,genres,userId,rating,timestamp
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,2.0,3.5,1.141416e+09
1,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,3.0,4.0,1.439472e+09
2,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,4.0,3.0,1.573944e+09
3,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,5.0,4.0,8.586259e+08
4,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,8.0,4.0,8.904925e+08


In [9]:
dataset = dataset.drop(columns = ["timestamp"], axis = 1)

In [10]:
dataset.head()

,movieId,title,genres,userId,rating
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,2.0,3.5
1,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,3.0,4.0
2,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,4.0,3.0
3,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,5.0,4.0
4,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,8.0,4.0


# Creating the SVD Model 

In [11]:
from surprise import Dataset, Reader, SVD

clean_dataset = dataset.dropna(subset = ['userId', 'movieId', 'rating']) # Handling missing values 

clean_dataset = clean_dataset.sample(n = 500000, random_state = 42)

reader = Reader(rating_scale = (0.5, 5.0))

data = Dataset.load_from_df(clean_dataset[["userId", "movieId", "rating"]], reader)

print("Successfully loaded the dataset")

Successfully loaded the dataset


In [13]:
from surprise import SVD
from surprise.model_selection import train_test_split
from surprise import accuracy

x, y = train_test_split(data, test_size = 0.2, random_state = 42)

algo = SVD()

algo.fit(x)

predictions = algo.test(y)

rmse = accuracy.rmse(predictions)
print(f"Baseline RMSE: {rmse}" )

RMSE: 0.9306
Baseline RMSE: 0.930580361241346


In [14]:
def get_top_n_recommendations(user_id, n = 10):
    
    unique_movies = dataset[['movieId', 'title']].drop_duplicates()
    
    movies_watched = dataset[dataset['userId'] == user_id]['movieId'].tolist()
    
    unseen_movies = unique_movies[~unique_movies['movieId'].isin(movies_watched)].copy()
    
    print(f"Predicting ratings for User {user_id}...")
    unseen_movies['predicted_rating'] = unseen_movies['movieId'].apply(lambda x: algo.predict(user_id, x).est)
    
    top_recommendations = unseen_movies.sort_values(by = 'predicted_rating', ascending = False).head(n)
    
    return top_recommendations[['title', 'predicted_rating']]

In [15]:
top_10 = get_top_n_recommendations(user_id = 3.0, n = 10)
print(top_10)

Predicting ratings for User 3.0...
                                                      title  predicted_rating
18683437                                  White Heat (1949)          4.631949
13414030                                 City Lights (1931)          4.611075
5214489       Sunset Blvd. (a.k.a. Sunset Boulevard) (1950)          4.610441
13653178                                Hustler, The (1961)          4.604259
6342845                                       Brazil (1985)          4.595859
17486631  Man with the Movie Camera, The (Chelovek s kin...          4.585121
18483621                         Pride and Prejudice (1995)          4.584475
14340044                              Love and Death (1975)          4.584421
12937083  Bicycle Thieves (a.k.a. The Bicycle Thief) (a....          4.581824
9401638                     In the Heat of the Night (1967)          4.575375


# Creating a Custom Neural Network Model

In [13]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split


user_encoder = LabelEncoder()  # For number conversion
movie_encoder = LabelEncoder()


clean_dataset['user_idx'] = user_encoder.fit_transform(clean_dataset['userId'])
clean_dataset['movie_idx'] = movie_encoder.fit_transform(clean_dataset['movieId'])


num_users = clean_dataset['user_idx'].nunique()  # Counting unique users and movies
num_movies = clean_dataset['movie_idx'].nunique()


train_df, temp_df = train_test_split(clean_dataset, test_size = 0.2, random_state = 42)
val_df, test_df = train_test_split(temp_df, test_size = 0.2, random_state = 42)


class MovieDataset(Dataset):
    def __init__(self, users, movies, ratings):
        self.users = torch.tensor(users.values, dtype = torch.long)  # Converting to tensors
        self.movies = torch.tensor(movies.values, dtype = torch.long)
        self.ratings = torch.tensor(ratings.values, dtype = torch.float32)
         
    def __len__(self):
        return len(self.users) # Returns how many rows are in the dataset
    
    def __getitem__(self, index):
        return self.users[index], self.movies[index], self.ratings[index]  # Returns the user, movie, and rating for a given position


train_dataset = MovieDataset(train_df['user_idx'], train_df['movie_idx'], train_df['rating'])
val_dataset = MovieDataset(val_df['user_idx'], val_df['movie_idx'], val_df['rating'])
test_dataset = MovieDataset(test_df['user_idx'], test_df['movie_idx'], test_df['rating'])


train_loader = DataLoader(train_dataset, batch_size = 1024, shuffle = True)
val_loader = DataLoader(val_dataset, batch_size = 1024, shuffle = False)
test_loader = DataLoader(test_dataset, batch_size = 1024, shuffle = False)

In [ ]:
class MovieRecommender(nn.Module):
    def __init__(self, num_users, num_movies, embedding_size = 32):
        super(MovieRecommender, self).__init__()
        
        self.user_embedding = nn.Embedding(num_users, embedding_size)  # For personalized preferences
        self.movie_embedding = nn.Embedding(num_movies, embedding_size)
        
        nn.init.normal_(self.user_embedding.weight, mean = 0, std = 0.01)
        nn.init.normal_(self.movie_embedding.weight, mean = 0, std = 0.01)
        
        self.user_bias = nn.Embedding(num_users, 1)  # To deal with extremes 
        self.movie_bias = nn.Embedding(num_movies, 1)
        
        self.dropout = nn.Dropout(p = 0.3)  # To prevent overfitting by randomly turning off neurons during training
        
        self.layer1 = nn.Linear(embedding_size * 2, 64) 
        self.bn1 = nn.BatchNorm1d(64)
        self.relu1 = nn.ReLU()
        self.layer2 = nn.Linear(64, 32)
        self.bn2= nn.BatchNorm1d(32)
        self.relu2 = nn.ReLU()
        
        self.output_layer = nn.Linear(32, 1) 
        
    def forward(self, user_index, movie_index):
        user_vector = self.user_embedding(user_index)
        movie_vector = self.movie_embedding(movie_index)
        
        combined_vector = torch.cat([user_vector, movie_vector], dim = 1)
        x = self.dropout(combined_vector)
        
        x = self.layer1(x)
        x = self.bn1(x)
        x = self.relu1(x)
        x = self.dropout(x) 
        
        x = self.layer2(x)
        x = self.bn2(x)
        x = self.relu2(x)
        x = self.dropout(x) 
        
        taste_prediction = self.output_layer(x)
        
        u_bias = self.user_bias(user_index)
        m_bias = self.movie_bias(movie_index)
        
        final_prediction = taste_prediction + u_bias + m_bias  # How compatible is the user with the movie + How generous is the user + How good is the movie is
        
        scaled_prediction = torch.sigmoid(final_prediction) * 4.5 + 0.5  # Squishing the output to be between 0.5 and 5.0
        
        return scaled_prediction.squeeze()

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


model = MovieRecommender(num_users = num_users, num_movies = num_movies, embedding_size = 32)
model = model.to(device)


criterion = nn.MSELoss() # Loss function 
optimizer = optim.Adam(model.parameters(), lr = 0.001, weight_decay = 1e-4)  # Optimizer


epochs = 20 


for epoch in range(epochs):
    model.train() # Setting the model for training
    total_train_loss = 0
    
    for users, movies, actual_ratings in train_loader:
        users, movies, actual_ratings = users.to(device), movies.to(device), actual_ratings.to(device) # Grabbing one batch of 1024 samples and moving them to the CPU/GPU
        
        optimizer.zero_grad()   # Gradient Resetting  
                               
        predicted_ratings = model(users, movies)            
        loss = criterion(predicted_ratings, actual_ratings) 
        
        loss.backward()  
                                           
        optimizer.step()  # Updating the weights based on gradients                                  
        
        total_train_loss += loss.item()
        
    train_rmse = (total_train_loss / len(train_loader)) ** 0.5
    
    model.eval()  # Setting the model for evaluation
    total_val_loss = 0 
    
    with torch.no_grad(): 
    
        for users, movies, actual_ratings in val_loader: 
            users, movies, actual_ratings = users.to(device), movies.to(device), actual_ratings.to(device)
            
            predicted_ratings = model(users, movies)
            loss = criterion(predicted_ratings, actual_ratings)
            
            total_val_loss += loss.item()
            
    val_rmse = (total_val_loss / len(val_loader)) ** 0.5 
    
    print(f"Epoch {epoch+1:02d}/{epochs} | Train RMSE: {train_rmse:.4f} | Val RMSE: {val_rmse:.4f}")

print("Training Complete")

Epoch 01/20 | Train RMSE: 1.2747 | Val RMSE: 1.0494
Epoch 02/20 | Train RMSE: 0.9803 | Val RMSE: 0.9968
Epoch 03/20 | Train RMSE: 0.8813 | Val RMSE: 0.9862
Epoch 04/20 | Train RMSE: 0.8328 | Val RMSE: 0.9803
Epoch 05/20 | Train RMSE: 0.8022 | Val RMSE: 0.9755
Epoch 06/20 | Train RMSE: 0.7816 | Val RMSE: 0.9777
Epoch 07/20 | Train RMSE: 0.7664 | Val RMSE: 0.9758
Epoch 08/20 | Train RMSE: 0.7545 | Val RMSE: 0.9812
Epoch 09/20 | Train RMSE: 0.7437 | Val RMSE: 0.9802
Epoch 10/20 | Train RMSE: 0.7336 | Val RMSE: 0.9879
Epoch 11/20 | Train RMSE: 0.7256 | Val RMSE: 0.9874
Epoch 12/20 | Train RMSE: 0.7167 | Val RMSE: 0.9946
Epoch 13/20 | Train RMSE: 0.7108 | Val RMSE: 0.9898
Epoch 14/20 | Train RMSE: 0.7034 | Val RMSE: 0.9912
Epoch 15/20 | Train RMSE: 0.6968 | Val RMSE: 0.9954
Epoch 16/20 | Train RMSE: 0.6908 | Val RMSE: 1.0009
Epoch 17/20 | Train RMSE: 0.6858 | Val RMSE: 0.9999
Epoch 18/20 | Train RMSE: 0.6806 | Val RMSE: 0.9971
Epoch 19/20 | Train RMSE: 0.6762 | Val RMSE: 1.0046
Epoch 20/20 

In [ ]:
model.eval()
total_test_loss = 0


with torch.no_grad():
    for users, movies, actual_ratings in test_loader:
        users, movies, actual_ratings = users.to(device), movies.to(device), actual_ratings.to(device)
        
        predicted_ratings = model(users, movies)
        loss = criterion(predicted_ratings, actual_ratings)
        
        total_test_loss += loss.item()
        
        
avg_test_loss = total_test_loss / len(test_loader)
test_rmse = avg_test_loss ** 0.5


print(f"Final Test RMSE: {test_rmse:.4f}")


if test_rmse < 0.9303:
    print("Model is ready")
else:
    print("Model has overfitted")

Final Test RMSE: 0.9936
Model has overfitted


Hyperparameter Tuning


In [28]:
class MovieRecommender(nn.Module):
    def __init__(self, num_users, num_movies, hidden_layers = 2, embedding_size = 32, dropout_rate = 0.3):
        super(MovieRecommender, self).__init__()
        
        self.user_embedding = nn.Embedding(num_users, embedding_size)
        self.movie_embedding = nn.Embedding(num_movies, embedding_size)
        
        nn.init.normal_(self.user_embedding.weight, mean = 0, std = 0.01)
        nn.init.normal_(self.movie_embedding.weight, mean = 0, std = 0.01)
        
        self.user_bias = nn.Embedding(num_users, 1)
        self.movie_bias = nn.Embedding(num_movies, 1)
        
        self.dropout = nn.Dropout(p = dropout_rate)
        
        layers = []
        input_size = embedding_size * 2
        
        for i in range(hidden_layers):
            layers.append(nn.Linear(input_size, 64))
            layers.append(nn.BatchNorm1d(64))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(p = dropout_rate))
            input_size = 64 
            
        self.hidden_network = nn.Sequential(*layers)  # * because Sequential needs layers individually and not as a list
        
        self.output_layer = nn.Linear(64, 1) 
        
    def forward(self, user_index, movie_index):
        user_vector = self.user_embedding(user_index)
        movie_vector = self.movie_embedding(movie_index)
        
        combined_vector = torch.cat([user_vector, movie_vector], dim = 1)
        x = self.dropout(combined_vector)
        
        x = self.hidden_network(x)
        
        taste_prediction = self.output_layer(x)
        
        u_bias = self.user_bias(user_index)
        m_bias = self.movie_bias(movie_index)
        
        final_prediction = taste_prediction + u_bias + m_bias
        scaled_prediction = torch.sigmoid(final_prediction) * 4.5 + 0.5 
        
        return scaled_prediction.squeeze()

In [32]:
import optuna

def objective(trial):
    hidden_layers = trial.suggest_int("hidden_layers", 1, 6) 
    embedding_size = trial.suggest_categorical("embedding_size", [16, 32, 64])
    dropout_rate = trial.suggest_float("dropout_rate", 0.1, 0.6)
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log = True)
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-3, log = True)
    epochs = trial.suggest_int("epochs", 10, 30, step = 5)
    
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    
    model = MovieRecommender(
        num_users = num_users, 
        num_movies = num_movies, 
        hidden_layers = hidden_layers,
        embedding_size = embedding_size, 
        dropout_rate = dropout_rate
    ).to(device)
    
    
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr = lr, weight_decay = weight_decay)
    
    
    for epoch in range(epochs):
        model.train()
        for users, movies, actual_ratings in train_loader:
            users, movies, actual_ratings = users.to(device), movies.to(device), actual_ratings.to(device)
            
            optimizer.zero_grad()
            predicted_ratings = model(users, movies)
            loss = criterion(predicted_ratings, actual_ratings)
            loss.backward()
            optimizer.step()
            

    model.eval()
    total_val_loss = 0
    
    
    with torch.no_grad():
        for users, movies, actual_ratings in val_loader:
            users, movies, actual_ratings = users.to(device), movies.to(device), actual_ratings.to(device)
            predicted_ratings = model(users, movies)
            loss = criterion(predicted_ratings, actual_ratings)
            total_val_loss += loss.item()
    
    
    val_rmse = (total_val_loss / len(val_loader)) ** 0.5
    return val_rmse

In [33]:
study = optuna.create_study(direction = "minimize", sampler = optuna.samplers.TPESampler())
study.optimize(objective, n_trials = 10)


print(f"Best Validation RMSE: {study.best_value:.4f}")
print("Best Hyperparameters: ")


for key, value in study.best_params.items():
    print(f"  {key}: {value}")

[I 2026-05-01 21:20:42,974] A new study created in memory with name: no-name-9e5ac42c-6223-47e5-97d9-77f24b2ac847
[I 2026-05-01 21:25:42,158] Trial 0 finished with value: 0.9558517456302743 and parameters: {'hidden_layers': 1, 'embedding_size': 64, 'dropout_rate': 0.42543907428799954, 'lr': 0.0019279191501920861, 'weight_decay': 9.965644257993738e-05, 'epochs': 20}. Best is trial 0 with value: 0.9558517456302743.
[I 2026-05-01 21:29:51,425] Trial 1 finished with value: 0.9918630379898459 and parameters: {'hidden_layers': 1, 'embedding_size': 16, 'dropout_rate': 0.3555379927093608, 'lr': 0.00392763116372863, 'weight_decay': 1.3573806656616324e-06, 'epochs': 25}. Best is trial 0 with value: 0.9558517456302743.
[I 2026-05-01 21:39:39,484] Trial 2 finished with value: 1.2908149510239946 and parameters: {'hidden_layers': 6, 'embedding_size': 32, 'dropout_rate': 0.5207208934367102, 'lr': 0.0001989613933187753, 'weight_decay': 0.0003573906316385491, 'epochs': 30}. Best is trial 0 with value: 

Best Validation RMSE: 0.9559
Best Hyperparameters: 
  hidden_layers: 1
  embedding_size: 64
  dropout_rate: 0.42543907428799954
  lr: 0.0019279191501920861
  weight_decay: 9.965644257993738e-05
  epochs: 20


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


final_model = MovieRecommender(
    num_users = num_users, 
    num_movies = num_movies, 
    hidden_layers = 1,             
    embedding_size = 64,           
    dropout_rate = 0.4254          
).to(device)


criterion = nn.MSELoss()
optimizer = optim.Adam(final_model.parameters(), lr = 0.0019, weight_decay = 9.96e-05)


epochs = 20


for epoch in range(epochs):
    final_model.train()
    total_train_loss = 0
    for users, movies, actual_ratings in train_loader:
        users, movies, actual_ratings = users.to(device), movies.to(device), actual_ratings.to(device)
        
        optimizer.zero_grad()
        predicted_ratings = final_model(users, movies)
        loss = criterion(predicted_ratings, actual_ratings)
        loss.backward()
        optimizer.step()
        total_train_loss += loss.item()
        
    train_rmse = (total_train_loss / len(train_loader)) ** 0.5
    print(f"Epoch {epoch+1:02d}/{epochs} | Train RMSE: {train_rmse:.4f}")


print("Training Complete")

Epoch 01/20 | Train RMSE: 1.1833
Epoch 02/20 | Train RMSE: 0.9429
Epoch 03/20 | Train RMSE: 0.8708
Epoch 04/20 | Train RMSE: 0.8406
Epoch 05/20 | Train RMSE: 0.8252
Epoch 06/20 | Train RMSE: 0.8165
Epoch 07/20 | Train RMSE: 0.8120
Epoch 08/20 | Train RMSE: 0.8085
Epoch 09/20 | Train RMSE: 0.8046
Epoch 10/20 | Train RMSE: 0.8031
Epoch 11/20 | Train RMSE: 0.8005
Epoch 12/20 | Train RMSE: 0.7969
Epoch 13/20 | Train RMSE: 0.7945
Epoch 14/20 | Train RMSE: 0.7926
Epoch 15/20 | Train RMSE: 0.7898
Epoch 16/20 | Train RMSE: 0.7872
Epoch 17/20 | Train RMSE: 0.7846
Epoch 18/20 | Train RMSE: 0.7821
Epoch 19/20 | Train RMSE: 0.7797
Epoch 20/20 | Train RMSE: 0.7775
Training Complete


In [36]:
final_model.eval()
total_test_loss = 0

with torch.no_grad():
    for users, movies, actual_ratings in test_loader:
        users, movies, actual_ratings = users.to(device), movies.to(device), actual_ratings.to(device)
        
        predicted_ratings = final_model(users, movies)
        loss = criterion(predicted_ratings, actual_ratings)
        
        total_test_loss += loss.item()
        
avg_test_loss = total_test_loss / len(test_loader)
test_rmse = avg_test_loss ** 0.5

print(f"Final Test RMSE: {test_rmse:.4f}")

if test_rmse < 0.9306: 
    print("Model is ready")
else:
    print("Model has overfitted")

Final Test RMSE: 0.9451
Model has overfitted
